In [4]:
# ============================================
# Multi-Model Token Profiler & Memory Math Engine
# Week 1 - PyTorch & LLM Mechanics
# Tech Prime Pvt Limited - Advanced AI/ML Internship
# ============================================

import torch
import math
from transformers import AutoTokenizer
import json
import csv
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. CONFIGURATION
# ============================================

class LLMConfig:
    """Configuration for LLM profiling"""
    
    # Supported models - using only publicly available tokenizers
    MODELS = {
        'gpt2': {
            'name': 'GPT-2',
            'tokenizer': 'gpt2',
            'params_b': 0.124,
            'layers': 12,
            'heads': 12,
            'head_dim': 64,
            'vocab_size': 50257,
            'max_ctx': 1024
        },
        'gpt2-medium': {
            'name': 'GPT-2 Medium',
            'tokenizer': 'gpt2-medium',
            'params_b': 0.35,
            'layers': 24,
            'heads': 16,
            'head_dim': 64,
            'vocab_size': 50257,
            'max_ctx': 1024
        },
        'gpt2-large': {
            'name': 'GPT-2 Large',
            'tokenizer': 'gpt2-large',
            'params_b': 0.77,
            'layers': 36,
            'heads': 20,
            'head_dim': 64,
            'vocab_size': 50257,
            'max_ctx': 1024
        },
        'bert-base': {
            'name': 'BERT Base',
            'tokenizer': 'bert-base-uncased',
            'params_b': 0.11,
            'layers': 12,
            'heads': 12,
            'head_dim': 64,
            'vocab_size': 30522,
            'max_ctx': 512
        },
        'roberta-base': {
            'name': 'RoBERTa Base',
            'tokenizer': 'roberta-base',
            'params_b': 0.125,
            'layers': 12,
            'heads': 12,
            'head_dim': 64,
            'vocab_size': 50265,
            'max_ctx': 512
        },
        't5-small': {
            'name': 'T5 Small',
            'tokenizer': 't5-small',
            'params_b': 0.06,
            'layers': 6,
            'heads': 8,
            'head_dim': 64,
            'vocab_size': 32128,
            'max_ctx': 512
        },
        'mistral-7b': {
            'name': 'Mistral-7B',
            'tokenizer': 'mistralai/Mistral-7B-v0.1',
            'params_b': 7,
            'layers': 32,
            'heads': 32,
            'head_dim': 128,
            'vocab_size': 32000,
            'max_ctx': 8192
        }
    }
    
    # LLaMA models (for VRAM calculation only, not tokenization)
    LLAMA_MODELS = {
        'llama-7b': {
            'name': 'LLaMA-7B',
            'params_b': 7,
            'layers': 32,
            'heads': 32,
            'head_dim': 128,
            'vocab_size': 32000,
            'max_ctx': 4096
        },
        'llama-13b': {
            'name': 'LLaMA-13B',
            'params_b': 13,
            'layers': 40,
            'heads': 40,
            'head_dim': 128,
            'vocab_size': 32000,
            'max_ctx': 4096
        },
        'llama-70b': {
            'name': 'LLaMA-70B',
            'params_b': 70,
            'layers': 80,
            'heads': 64,
            'head_dim': 128,
            'vocab_size': 32000,
            'max_ctx': 4096
        }
    }
    
    # Precision formats
    PRECISIONS = {
        'fp32': {'bits': 32, 'bytes': 4, 'name': 'FP32'},
        'fp16': {'bits': 16, 'bytes': 2, 'name': 'FP16'},
        'bf16': {'bits': 16, 'bytes': 2, 'name': 'BF16'},
        'int8': {'bits': 8, 'bytes': 1, 'name': 'INT8'},
        'int4': {'bits': 4, 'bytes': 0.5, 'name': 'INT4'}
    }
    
    # Default settings
    DEFAULT_BATCH_SIZE = 1
    DEFAULT_SEQ_LEN = 2048
    DEFAULT_PRECISION = 'fp16'

# ============================================
# 2. TOKEN PROFILER
# ============================================

class TokenProfiler:
    """Profile tokenization across different models"""
    
    def __init__(self):
        self.tokenizers = {}
        self.load_tokenizers()
    
    def load_tokenizers(self):
        """Load tokenizers for all models"""
        print("Loading tokenizers...")
        for model_key, config in LLMConfig.MODELS.items():
            try:
                tokenizer = AutoTokenizer.from_pretrained(
                    config['tokenizer'],
                    trust_remote_code=True
                )
                self.tokenizers[model_key] = tokenizer
                print(f"  ✓ Loaded: {config['name']}")
            except Exception as e:
                print(f"  ✗ Failed to load {config['name']}: {e}")
    
    def count_tokens(self, text: str, model_key: str = 'gpt2') -> Dict:
        """Count tokens for a given text and model"""
        if model_key not in self.tokenizers:
            # Try fallback to a model that is loaded
            available_models = list(self.tokenizers.keys())
            if available_models:
                fallback = available_models[0]
                print(f"  ⚠ Model {model_key} not loaded, using {fallback} as fallback")
                model_key = fallback
            else:
                raise ValueError("No tokenizers loaded. Please check your internet connection.")
        
        tokenizer = self.tokenizers[model_key]
        
        try:
            tokens = tokenizer.encode(text)
            token_strings = tokenizer.convert_ids_to_tokens(tokens)
            
            return {
                'text': text[:100] + '...' if len(text) > 100 else text,
                'model': LLMConfig.MODELS.get(model_key, {}).get('name', model_key),
                'token_count': len(tokens),
                'token_ids': tokens[:10],  # First 10 tokens
                'token_strings': token_strings[:10],
                'vocab_size': tokenizer.vocab_size
            }
        except Exception as e:
            return {
                'error': str(e),
                'model': model_key
            }
    
    def profile_document(self, document: str, models: Optional[List[str]] = None) -> Dict:
        """Profile a document across multiple models"""
        if models is None:
            models = list(self.tokenizers.keys())
        
        results = {}
        for model_key in models:
            try:
                results[model_key] = self.count_tokens(document, model_key)
            except Exception as e:
                results[model_key] = {'error': str(e)}
        
        return results
    
    def estimate_cost(self, token_count: int, cost_per_1k: float = 0.001) -> float:
        """Estimate API cost based on token count"""
        return (token_count / 1000) * cost_per_1k
    
    def generate_report(self, text: str) -> str:
        """Generate a report of tokenization across models"""
        results = self.profile_document(text)
        
        report = [
            "=" * 70,
            "TOKENIZATION REPORT",
            "=" * 70,
            f"\nText Sample: {text[:200]}...\n",
            f"{'Model':<20} {'Tokens':<10} {'Vocab Size':<15} {'Est. Cost (1000x)':<15}"
        ]
        report.append("-" * 70)
        
        for model_key, result in results.items():
            if 'error' in result:
                report.append(f"{model_key:<20} {'ERROR':<10}")
            else:
                cost = self.estimate_cost(result['token_count'])
                report.append(
                    f"{result['model']:<20} "
                    f"{result['token_count']:<10} "
                    f"{result['vocab_size']:<15} "
                    f"${cost*1000:<14.4f}"
                )
        
        report.append("=" * 70)
        return "\n".join(report)

# ============================================
# 3. MEMORY MATH ENGINE
# ============================================

class MemoryMathEngine:
    """Calculate VRAM requirements for LLM inference"""
    
    def __init__(self):
        self.model_configs = {**LLMConfig.MODELS, **LLMConfig.LLAMA_MODELS}
        self.precisions = LLMConfig.PRECISIONS
    
    def calculate_model_weights(self, params_b: float, precision: str) -> Dict:
        """Calculate memory for model weights"""
        if precision not in self.precisions:
            raise ValueError(f"Precision {precision} not supported")
        
        bytes_per_param = self.precisions[precision]['bytes']
        total_bytes = params_b * 1e9 * bytes_per_param
        
        return {
            'bytes': total_bytes,
            'MB': total_bytes / (1024 * 1024),
            'GB': total_bytes / (1024**3)
        }
    
    def calculate_kv_cache(self, batch_size: int, seq_len: int, 
                           layers: int, heads: int, head_dim: int,
                           precision: str) -> Dict:
        """Calculate KV-cache memory"""
        if precision not in self.precisions:
            raise ValueError(f"Precision {precision} not supported")
        
        bytes_per_param = self.precisions[precision]['bytes']
        
        # Each layer stores K and V
        per_layer_bytes = 2 * batch_size * heads * seq_len * head_dim * bytes_per_param
        total_bytes = per_layer_bytes * layers
        
        return {
            'bytes': total_bytes,
            'MB': total_bytes / (1024 * 1024),
            'GB': total_bytes / (1024**3),
            'formula': f"2 × {batch_size} × {heads} × {seq_len} × {head_dim} × {layers} × {bytes_per_param}B"
        }
    
    def calculate_activations(self, params_b: float, seq_len: int, precision: str) -> Dict:
        """Estimate activation memory"""
        # Rough estimate: 20-30% of model weights
        factor = 0.25
        weights = self.calculate_model_weights(params_b, precision)
        
        total_bytes = weights['bytes'] * factor
        
        return {
            'bytes': total_bytes,
            'MB': total_bytes / (1024 * 1024),
            'GB': total_bytes / (1024**3),
            'factor': factor
        }
    
    def calculate_total_vram(self, model_key: str, seq_len: int, 
                           batch_size: int, precision: str = 'fp16') -> Dict:
        """Calculate total VRAM requirements"""
        if model_key not in self.model_configs:
            raise ValueError(f"Model {model_key} not found")
        
        config = self.model_configs[model_key]
        
        # Model weights
        weights = self.calculate_model_weights(config['params_b'], precision)
        
        # KV-cache
        kv_cache = self.calculate_kv_cache(
            batch_size, seq_len,
            config['layers'],
            config['heads'],
            config['head_dim'],
            precision
        )
        
        # Activations
        activations = self.calculate_activations(config['params_b'], seq_len, precision)
        
        # Gradients (during training)
        gradients = {'bytes': weights['bytes'], 'GB': weights['GB']}
        
        # Total
        total_bytes = weights['bytes'] + kv_cache['bytes'] + activations['bytes']
        total_gb = total_bytes / (1024**3)
        
        return {
            'model_name': config['name'],
            'params_b': config['params_b'],
            'seq_len': seq_len,
            'batch_size': batch_size,
            'precision': precision,
            'model_weights': weights,
            'kv_cache': kv_cache,
            'activations': activations,
            'gradients': gradients,
            'total_gb': total_gb,
            'total_bytes': total_bytes
        }
    
    def compare_deployments(self, seq_len: int, batch_size: int = 1):
        """Compare VRAM across models and precisions"""
        
        models = ['llama-7b', 'llama-13b', 'llama-70b', 'mistral-7b']
        precisions = ['fp16', 'int8', 'int4']
        
        print("VRAM Comparison for Deployment")
        print("=" * 80)
        print(f"Sequence Length: {seq_len}, Batch Size: {batch_size}")
        print("-" * 80)
        print(f"{'Model':<15} {'Precision':<10} {'Weights':<12} {'KV-Cache':<12} {'Total':<12}")
        print("-" * 80)
        
        results = []
        for model in models:
            if model not in self.model_configs:
                continue
            for prec in precisions:
                try:
                    mem = self.calculate_total_vram(model, seq_len, batch_size, prec)
                    results.append({
                        'model': mem['model_name'],
                        'precision': prec,
                        'weights_gb': mem['model_weights']['GB'],
                        'kv_cache_gb': mem['kv_cache']['GB'],
                        'total_gb': mem['total_gb']
                    })
                    print(f"{mem['model_name']:<15} {prec:<10} "
                          f"{mem['model_weights']['GB']:<12.2f} "
                          f"{mem['kv_cache']['GB']:<12.2f} "
                          f"{mem['total_gb']:<12.2f}")
                except Exception as e:
                    print(f"{model:<15} {prec:<10} {'ERROR':<12}")
        
        print("=" * 80)
        return results

# ============================================
# 4. COST ESTIMATOR
# ============================================

class CostEstimator:
    """Estimate API costs for LLM inference"""
    
    def __init__(self):
        # Standard API pricing (per 1M tokens)
        self.pricing = {
            'gpt-4': {
                'input': 30.00,
                'output': 60.00
            },
            'gpt-4-turbo': {
                'input': 10.00,
                'output': 30.00
            },
            'gpt-3.5-turbo': {
                'input': 0.50,
                'output': 1.50
            },
            'claude-3-opus': {
                'input': 15.00,
                'output': 75.00
            },
            'claude-3-sonnet': {
                'input': 3.00,
                'output': 15.00
            },
            'llama-2-7b': {
                'input': 0.20,
                'output': 0.20
            },
            'llama-2-13b': {
                'input': 0.50,
                'output': 0.50
            },
            'llama-2-70b': {
                'input': 2.00,
                'output': 2.00
            },
            'mistral-7b': {
                'input': 0.20,
                'output': 0.20
            },
            'mixtral-8x7b': {
                'input': 0.50,
                'output': 0.50
            }
        }
    
    def estimate_cost(self, token_count: int, model: str, 
                     input_tokens: int = None, output_tokens: int = None) -> Dict:
        """Estimate cost for a request"""
        
        if model not in self.pricing:
            raise ValueError(f"Model {model} not found in pricing")
        
        if input_tokens is None:
            input_tokens = token_count
            output_tokens = token_count * 0.5  # Assume output is half of input
        
        pricing = self.pricing[model]
        
        input_cost = (input_tokens / 1_000_000) * pricing['input']
        output_cost = (output_tokens / 1_000_000) * pricing['output']
        total_cost = input_cost + output_cost
        
        return {
            'model': model,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'total_tokens': input_tokens + output_tokens,
            'input_cost': input_cost,
            'output_cost': output_cost,
            'total_cost': total_cost
        }
    
    def batch_cost_estimate(self, requests: List[Dict]) -> Dict:
        """Estimate cost for a batch of requests"""
        total_input_tokens = 0
        total_output_tokens = 0
        total_cost = 0
        
        results = []
        for req in requests:
            cost = self.estimate_cost(
                req.get('token_count', 0),
                req['model'],
                req.get('input_tokens'),
                req.get('output_tokens')
            )
            results.append(cost)
            total_input_tokens += cost['input_tokens']
            total_output_tokens += cost['output_tokens']
            total_cost += cost['total_cost']
        
        return {
            'total_requests': len(requests),
            'total_input_tokens': total_input_tokens,
            'total_output_tokens': total_output_tokens,
            'total_tokens': total_input_tokens + total_output_tokens,
            'total_cost': total_cost,
            'per_request': results
        }
    
    def monthly_cost_projection(self, daily_tokens: int, model: str,
                               output_ratio: float = 0.5) -> Dict:
        """Project monthly costs based on daily usage"""
        monthly_tokens = daily_tokens * 30
        
        input_tokens = int(monthly_tokens / (1 + output_ratio))
        output_tokens = int(monthly_tokens - input_tokens)
        
        cost = self.estimate_cost(
            monthly_tokens,
            model,
            input_tokens,
            output_tokens
        )
        
        return {
            'daily_tokens': daily_tokens,
            'monthly_tokens': monthly_tokens,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'monthly_cost': cost['total_cost'],
            'daily_cost': cost['total_cost'] / 30,
            'model': model
        }

# ============================================
# 5. MAIN APPLICATION
# ============================================

class LLMProfiler:
    """Main application for LLM profiling"""
    
    def __init__(self):
        self.token_profiler = TokenProfiler()
        self.memory_engine = MemoryMathEngine()
        self.cost_estimator = CostEstimator()
    
    def profile_pipeline(self, text: str, model_key: str = 'gpt2',
                        seq_len: int = 2048, precision: str = 'fp16'):
        """Run complete profiling pipeline"""
        
        print("=" * 80)
        print("LLM PROFILING PIPELINE")
        print("=" * 80)
        
        # 1. Tokenization
        print("\n1. TOKENIZATION")
        print("-" * 40)
        token_result = self.token_profiler.count_tokens(text, model_key)
        if 'error' in token_result:
            print(f"Error: {token_result['error']}")
            return
        print(f"Model: {token_result['model']}")
        print(f"Token Count: {token_result['token_count']}")
        print(f"Vocab Size: {token_result['vocab_size']}")
        
        # 2. Memory Estimation
        print("\n2. MEMORY ESTIMATION")
        print("-" * 40)
        try:
            mem_result = self.memory_engine.calculate_total_vram(
                model_key, seq_len, 1, precision
            )
            print(f"Model: {mem_result['model_name']}")
            print(f"Parameters: {mem_result['params_b']}B")
            print(f"Precision: {mem_result['precision']}")
            print(f"Model Weights: {mem_result['model_weights']['GB']:.2f} GB")
            print(f"KV-Cache: {mem_result['kv_cache']['GB']:.2f} GB")
            print(f"Activations: {mem_result['activations']['GB']:.2f} GB")
            print(f"Total VRAM: {mem_result['total_gb']:.2f} GB")
        except Exception as e:
            print(f"Error in memory estimation: {e}")
            mem_result = None
        
        # 3. Cost Estimation
        print("\n3. COST ESTIMATION")
        print("-" * 40)
        try:
            cost = self.cost_estimator.estimate_cost(
                token_result['token_count'],
                'gpt-3.5-turbo'
            )
            print(f"Model: {cost['model']}")
            print(f"Input Tokens: {cost['input_tokens']:,}")
            print(f"Output Tokens: {cost['output_tokens']:,}")
            print(f"Total Cost: ${cost['total_cost']:.6f}")
        except Exception as e:
            print(f"Error in cost estimation: {e}")
            cost = None
        
        # 4. Monthly Projection
        print("\n4. MONTHLY PROJECTION")
        print("-" * 40)
        try:
            monthly = self.cost_estimator.monthly_cost_projection(
                token_result['token_count'] * 100,  # 100 requests per day
                'gpt-3.5-turbo'
            )
            print(f"Daily Requests: 100")
            print(f"Daily Tokens: {monthly['daily_tokens']:,}")
            print(f"Monthly Tokens: {monthly['monthly_tokens']:,}")
            print(f"Monthly Cost: ${monthly['monthly_cost']:.2f}")
        except Exception as e:
            print(f"Error in monthly projection: {e}")
            monthly = None
        
        print("\n" + "=" * 80)
        
        return {
            'tokenization': token_result,
            'memory': mem_result,
            'cost': cost,
            'monthly': monthly
        }
    
    def generate_full_report(self, text: str, models: List[str] = None,
                           seq_lens: List[int] = None):
        """Generate comprehensive report"""
        
        if models is None:
            models = ['llama-7b', 'llama-13b', 'llama-70b', 'mistral-7b']
        
        if seq_lens is None:
            seq_lens = [512, 1024, 2048, 4096]
        
        print("=" * 80)
        print("COMPREHENSIVE LLM REPORT")
        print("=" * 80)
        
        # Tokenization report
        print("\nTOKENIZATION ACROSS MODELS")
        print("-" * 40)
        token_report = self.token_profiler.generate_report(text)
        print(token_report)
        
        # VRAM comparison
        print("\nVRAM COMPARISON")
        print("-" * 40)
        for seq_len in seq_lens:
            print(f"\nSequence Length: {seq_len}")
            self.memory_engine.compare_deployments(seq_len)
        
        # Cost comparison
        print("\nCOST COMPARISON")
        print("-" * 40)
        models_cost = ['gpt-3.5-turbo', 'gpt-4', 'claude-3-sonnet', 'llama-2-70b']
        print(f"{'Model':<25} {'Cost per 1M tokens (avg)':<25} {'Cost per 1k tokens':<20}")
        print("-" * 70)
        for model in models_cost:
            pricing = self.cost_estimator.pricing.get(model, {})
            if pricing:
                avg_cost = (pricing['input'] + pricing['output']) / 2
                print(f"{model:<25} ${avg_cost:<24.2f} ${avg_cost/1000:<19.4f}")
        
        print("\n" + "=" * 80)

# ============================================
# 6. RUN
# ============================================

def main():
    """Main entry point"""
    
    print("=" * 80)
    print("MULTI-MODEL TOKEN PROFILER & MEMORY MATH ENGINE")
    print("Tech Prime Pvt Limited - Advanced AI/ML Internship")
    print("=" * 80)
    
    # Initialize profiler
    profiler = LLMProfiler()
    
    # Sample text
    sample_text = """
    The Transformer architecture has revolutionized natural language processing.
    At its core, the self-attention mechanism allows the model to weigh the 
    importance of different parts of the input sequence. This enables the model
    to capture long-range dependencies and contextual relationships that were
    previously difficult for traditional recurrent neural networks.
    
    The key innovation is the attention mechanism, which computes a weighted
    sum of values based on the compatibility of queries and keys. This allows
    the model to focus on relevant parts of the input when generating each
    output token.
    """
    
    # Run profiling using a working model
    profiler.profile_pipeline(sample_text, model_key='gpt2')
    
    # Generate full report
    profiler.generate_full_report(sample_text)
    
    print("\n" + "=" * 80)
    print("✓ PROFILING COMPLETED SUCCESSFULLY!")
    print("=" * 80)

if __name__ == "__main__":
    main()

MULTI-MODEL TOKEN PROFILER & MEMORY MATH ENGINE
Tech Prime Pvt Limited - Advanced AI/ML Internship
Loading tokenizers...
  ✓ Loaded: GPT-2
  ✓ Loaded: GPT-2 Medium
  ✓ Loaded: GPT-2 Large
  ✓ Loaded: BERT Base
  ✓ Loaded: RoBERTa Base
  ✓ Loaded: T5 Small
  ✓ Loaded: Mistral-7B
LLM PROFILING PIPELINE

1. TOKENIZATION
----------------------------------------
Model: GPT-2
Token Count: 149
Vocab Size: 50257

2. MEMORY ESTIMATION
----------------------------------------
Model: GPT-2
Parameters: 0.124B
Precision: fp16
Model Weights: 0.23 GB
KV-Cache: 0.07 GB
Activations: 0.06 GB
Total VRAM: 0.36 GB

3. COST ESTIMATION
----------------------------------------
Model: gpt-3.5-turbo
Input Tokens: 149
Output Tokens: 74.5
Total Cost: $0.000186

4. MONTHLY PROJECTION
----------------------------------------
Daily Requests: 100
Daily Tokens: 14,900
Monthly Tokens: 447,000
Monthly Cost: $0.37

COMPREHENSIVE LLM REPORT

TOKENIZATION ACROSS MODELS
----------------------------------------
TOKENIZATION 